# Task 1 — EDA & Preprocessing

Explore the CFPB complaint dataset, filter to the four target products, clean the narratives, and save `data/processed/filtered_complaints.csv`.

In [1]:
import sys
from pathlib import Path

# Make the project root importable so `import src...` works inside the notebook
sys.path.append(str(Path.cwd().parent))

import matplotlib.pyplot as plt
import seaborn as sns

from src import data_processing as dp

sns.set_theme(style="whitegrid")

## 1. Load the full dataset

Download the full CFPB dataset and place the CSV in `data/raw/`, then point `RAW_PATH` at it.

In [ ]:
RAW_PATH = dp.RAW_DATA_DIR / "complaints.csv" 

df = dp.load_complaints(RAW_PATH)
print(f"Shape: {df.shape}")
df.head()

## 2. Initial EDA

### 2a. Distribution of complaints across products

In [ ]:
product_counts = dp.product_distribution(df)
print(product_counts)

plt.figure(figsize=(10, 6))
product_counts.head(15).plot(kind="barh")
plt.gca().invert_yaxis()
plt.title("Complaints per product (top 15)")
plt.xlabel("Number of complaints")
plt.tight_layout()
plt.show()

### 2b. Consumer narrative length (word count)

In [ ]:
df = dp.add_narrative_length(df)
print(df["narrative_word_count"].describe())

plt.figure(figsize=(10, 5))
sns.histplot(df.loc[df["narrative_word_count"] > 0, "narrative_word_count"], bins=60)
plt.title("Distribution of narrative length (words, narratives only)")
plt.xlabel("Word count")
plt.xlim(0, 1000)
plt.tight_layout()
plt.show()

# Flag very short and very long narratives
short = df[(df["narrative_word_count"] > 0) & (df["narrative_word_count"] < 5)]
long = df[df["narrative_word_count"] > 1000]
print(f"Very short (<5 words): {len(short):,}")
print(f"Very long (>1000 words): {len(long):,}")

### 2c. Complaints with vs. without narratives

In [ ]:
presence = dp.narrative_presence(df)
print(presence)

plt.figure(figsize=(5, 5))
plt.pie(presence.values(), labels=presence.keys(), autopct="%1.1f%%")
plt.title("Complaints with vs. without a narrative")
plt.show()

## 3. Filter the dataset

Keep only the four target products and drop empty narratives.

In [ ]:
filtered = dp.filter_complaints(df)
print(f"Rows after filtering: {len(filtered):,}")
filtered["product_category"].value_counts()

## 4. Clean the narratives

Lowercase, strip CFPB redactions (`XXXX`), remove boilerplate openers and special characters.

In [ ]:
cleaned = dp.clean_narratives(filtered)
print(f"Rows after cleaning: {len(cleaned):,}")

# Before / after example
sample = cleaned.iloc[0]
print("RAW:\n", sample[dp.NARRATIVE_COL][:300])
print("\nCLEANED:\n", sample["cleaned_narrative"][:300])

## 5. Save the cleaned, filtered dataset

In [ ]:
out_path = dp.save_filtered(cleaned)
print(f"Saved {len(cleaned):,} complaints to {out_path}")

## EDA Summary (write 2–3 paragraphs for the report)

- **Product distribution:** ...
- **Narrative length:** ...
- **Missing narratives:** ...
- **Cleaning decisions:** ...